# Bangladesh Fuel Observatory — Reproducible Research Pipeline

This notebook mirrors the production pipeline used by the website. It treats official retail prices as discrete events and market benchmarks as the information available before each event. The main evaluation is time ordered; random train/test shuffling is intentionally avoided.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.build_model import build, eligible_events, expanding_primary

market = pd.read_csv(ROOT / 'data/market_history.csv', parse_dates=['date'])
official = pd.read_csv(ROOT / 'data/official_prices.csv', parse_dates=['effective_date'])
config = json.loads((ROOT / 'data/config.json').read_text())
print('Market rows:', len(market))
print('Official events:', len(official))
print('Window:', config['policy_window_days'], 'calendar days')

## 1. Coverage and benchmark mapping

In [ ]:
print(market[['date','gasoil_10ppm','ron92','ron95','usd_bdt']].head())
print(market[['date','gasoil_10ppm','ron92','ron95','usd_bdt']].tail())
print('Benchmark counts:', market[['gasoil_10ppm','ron92','ron95']].notna().sum().to_dict())

## 2. International benchmark equivalent

For each market observation, benchmark USD/bbl is multiplied by USD/BDT and divided by 158.9873 litres/barrel.

In [ ]:
BARREL = config['barrel_litres']
for fuel, benchmark in config['benchmark_mapping'].items():
    series = market[benchmark] * market['usd_bdt'] / BARREL
    print(fuel, 'latest market-equivalent Tk/L =', round(series.dropna().iloc[-1], 2))

## 3. Official-event eligibility audit

An event is eligible when at least four valid market/FX observations exist in the prior 30 calendar days. The event-day market observation is excluded.

In [ ]:
for fuel in ['diesel','petrol','octane']:
    e = eligible_events(market, official, fuel)
    print('\n', fuel)
    display(e[['effective_date','official_price','window_observations','eligible','eligibility_reason']])

## 4. Expanding-calibration estimator

The primary historical prediction is the prior market-window signal plus the median domestic residual from earlier eligible events only. This is a no-lookahead baseline, not a claim that the specification is optimal.

In [ ]:
for fuel in ['diesel','petrol','octane']:
    e = eligible_events(market, official, fuel)
    bt = expanding_primary(e)
    print('\n', fuel)
    display(bt[['effective_date','official_price','market_signal','calibration_median','primary_pred','naive_pred']])

## 5. Market history visualization

In [ ]:
for fuel, benchmark in config['benchmark_mapping'].items():
    s = market.dropna(subset=[benchmark, 'usd_bdt']).copy()
    s['tk_l'] = s[benchmark] * s['usd_bdt'] / BARREL
    plt.figure(figsize=(9,4))
    plt.plot(s['date'], s['tk_l'])
    plt.title(f'{fuel.title()} — international benchmark equivalent')
    plt.ylabel('Tk/L')
    plt.xlabel('Date')
    plt.tight_layout()
    plt.show()

## 6. Production model output

Run the same build function used by `scripts/build_model.py` and inspect the JSON stored for the website.

In [ ]:
model = build()
for fuel in ['diesel','petrol','octane']:
    f = model['fuels'][fuel]
    print(fuel.title(), 'estimate:', f['estimated_retail_tk_l'], 'official reference:', f['official_reference_tk_l'], 'cutoff:', f['model_cutoff_date'])

## Interpretation

The current bundled sample has enough market history for a meaningful event-eligibility demonstration, but the official-event sample is still small. Any regression below the reporting threshold is exploratory; walk-forward predictions and prediction intervals should not be presented as mature evidence until their configured sample thresholds are met.